In [1]:
import numpy as np
import pandas as pd
from joblib import load
from utils.paths import DATASETS_PREPROCESSED_DIR, PROJECT_ROOT
import dieboldmariano

pp_models_dir = PROJECT_ROOT / "power_prediction" / "models"
ar_models_dir = PROJECT_ROOT / "autoregression-models" / "models"

alpha = 0.05

In [2]:
TARGET_LAGS = [1, 2, 3, 4, 5, 6, 24, 48]
FEATURE_LAGS = {
    "temperature_2m": [1, 17],
    "relative_humidity_2m": [6, 30],
    "wind_speed_10m": [6, 30],
    "wind_gusts_10m": [6, 30],
}


def add_lags(df, horizon):
    lagged = df.copy()
    lagged["hour"] = pd.to_datetime(lagged["time"]).dt.hour
    lagged["dayofweek"] = pd.to_datetime(lagged["time"]).dt.dayofweek
    lagged["month"] = pd.to_datetime(lagged["time"]).dt.month
    lagged["row_idx"] = np.arange(len(lagged))

    for lag in TARGET_LAGS:
        lagged[f"delta_p_lag_{lag}"] = lagged["delta_p"].shift(lag)

    for feature, lags in FEATURE_LAGS.items():
        for lag in lags:
            lagged[f"{feature}_lag_{lag}"] = lagged[feature].shift(lag)

    lagged["target"] = lagged["delta_p"].shift(-horizon)
    lagged = lagged.dropna().reset_index(drop=True)

    return lagged

In [3]:
rows = []

for loc in range(1, 5):
    df = pd.read_csv(DATASETS_PREPROCESSED_DIR / f"Location_{loc}_test.csv")

    X_pp = df.drop(columns=["time", "p_base", "delta_p"])
    y_pp_all = load(pp_models_dir / f"stacking_model_loc{loc}.pkl").predict(X_pp)

    for horizon in [1, 24, 48]:
        lagged = add_lags(df, horizon)
        # ar_model = load(ar_models_dir / f"stacking_regressor_h{horizon}.pkl")
        ar_model = load(ar_models_dir / f"hybrid_h{horizon}_s10.pkl")

        X_ar = lagged.drop(columns=["time", "delta_p", "target", "p_base", "row_idx"])
        y = lagged["target"].to_numpy()
        y_ar = ar_model.predict(X_ar)
        y_pp = np.asarray(y_pp_all)[lagged["row_idx"].to_numpy() + horizon]  # getting only the predictions corresponding to the lagged rows

        dm_stat, p_value = dieboldmariano.dm_test(y, y_ar, y_pp, h=horizon)

        rows.append({
            "Lokalizacja": f"{loc}", 
            "Horyzont": horizon, 
            "Statystyka": dm_stat, 
            "p_value": p_value, 
            "Odrzucenie H0": p_value < alpha
            })

pd.DataFrame(rows)

,Lokalizacja,Horyzont,Statystyka,p_value,Odrzucenie H0
0,1,1,-41.114141,0.000000e+00,True
1,1,24,2.946080,3.226831e-03,True
2,1,48,3.781561,1.568959e-04,True
3,2,1,-37.560698,2.671643e-286,True
4,2,24,5.129893,2.961720e-07,True
5,2,48,5.947767,2.823619e-09,True
6,3,1,-49.008245,0.000000e+00,True
7,3,24,2.806742,5.015679e-03,True
8,3,48,1.849929,6.435777e-02,False
9,4,1,-27.338888,5.922991e-158,True
